In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import mcstasscript as ms
import mcstastox as mx
import scipp as sc
import plopp as pp
from scippneutron.conversion.graph.beamline import beamline

parent = os.path.dirname(os.getcwd())
sys.path.append(parent)

from trex_reduction import (
    generate_bins,
    inelastic,
    _calc_pulse_centroid,
)  # , produce_trex_event_object

In [2]:
def produce_trex_event_object(
    event_object, data_path, monitor_name, centroids=None, to_s=1e-6
):
    """ """

    with mx.Read(data_path) as loaded_data:
        monitor_position = loaded_data.get_global_component_coordinates(monitor_name)

    data = ms.load_data(data_path)
    monitor = ms.name_search(monitor_name, data)

    event_object.coords["monitor_position"] = sc.vector(
        value=monitor_position, unit="m"
    )

    if centroids is None:
        centroids = _calc_pulse_centroid(monitor)

    look_up_tab = sc.DataArray(data=centroids, coords={"tof": centroids})

    tof_to_centroid = sc.lookup(look_up_tab, mode="previous")
    event_object = event_object.transform_coords(time_on_monitor=tof_to_centroid)
    event_object.coords["monitor_counts"] = sc.scalar(
        value=monitor.Ncount.sum(), unit="counts"
    )

    return event_object

In [3]:
file_path = parent + "/runs/LET_vanad"

with mx.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp(
        source_name="SourceMantid",
        sample_name="iso_samp",
    )

data = ms.load_data(file_path)

events = scipp_object["events"]
events

<scipp.DataArray>
Dimensions: Sizes[pixel_id:25886, ]
Coordinates:
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 30053, 30054]
* position                  vector3              [m]  (pixel_id)  [(-2.23064, -1.98529, 27.6971), (-2.19208, -1.98529, 27.7285), ..., (2.3436, 1.98529, 22.4005), (2.30641, 1.98529, 22.3674)]
* sample_position           vector3              [m]  ()  (0, 0, 25)
* source_position           vector3              [m]  ()  (0, 0, 0)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 59921),
          data=float64[counts],
          coords={'t':float64[s]})

In [4]:
# McStas provides absolute time, not time of flight
events.bins.coords["tof"] = events.bins.coords["t"]
# Add additional information required for inelastic scattering
events = produce_trex_event_object(events, file_path, "Monitor6")
qens_graph = {**beamline(scatter=True), **inelastic}
events = events.transform_coords(["Q", "mag_Q", "dE"], graph=qens_graph)

# create (qx, qy, qz, en)
# Note: Need to use copy() to allow for binning later
events.bins.coords["qx"] = events.bins.coords["Q"].fields.x.copy()
events.bins.coords["qy"] = events.bins.coords["Q"].fields.y.copy()
events.bins.coords["qz"] = events.bins.coords["Q"].fields.z.copy()
events.bins.coords["en"] = events.bins.coords["dE"]

events

<scipp.DataArray>
Dimensions: Sizes[pixel_id:25886, ]
Coordinates:
  L1                        float64              [m]  ()  25
  L2                        float64              [m]  (pixel_id)  [4.02385, 4.02385, ..., 4.02385, 4.02385]
  Lm                        float64              [m]  ()  23.505
  incident_beam             vector3              [m]  ()  (0, 0, 25)
  monitor_beam              vector3              [m]  ()  (0, 0, 23.505)
* monitor_counts            float64         [counts]  ()  1.65719e+06
  monitor_position          vector3              [m]  ()  (0, 0, 23.505)
* pixel_id                    int64  [dimensionless]  (pixel_id)  [0, 1, ..., 30053, 30054]
  position                  vector3              [m]  (pixel_id)  [(-2.23064, -1.98529, 27.6971), (-2.19208, -1.98529, 27.7285), ..., (2.3436, 1.98529, 22.4005), (2.30641, 1.98529, 22.3674)]
  sample_position           vector3              [m]  ()  (0, 0, 25)
  scattered_beam            vector3              [m]  (pixel_id)  [(-2.23064, -1.98529, 2.69708), (-2.19208, -1.98529, 2.72851), ..., (2.3436, 1.98529, -2.59953), (2.30641, 1.98529, -2.63258)]
  source_position           vector3              [m]  ()  (0, 0, 0)
  unit_kf                   vector3  [dimensionless]  (pixel_id)  [(-0.554355, -0.493381, 0.670273), (-0.544771, -0.493381, 0.678085), ..., (0.582427, 0.493381, -0.646029), (0.573185, 0.493381, -0.654243)]
  unit_ki                   vector3  [dimensionless]  ()  (0, 0, 1)
Data:
                          DataArrayView        <no unit>  (pixel_id)  binned data: dim='events', content=DataArray(
          dims=(events: 59921),
          data=float64[counts],
          coords={'t':float64[s], 'tof':float64[s], 'time_on_monitor':float64[s],
                  'vi':float64[m/s], 'time_on_sample':float64[s], 'mag_ki':float64[1/Å],
                  'vf':float64[m/s], 'ki':vector3[1/Å], 'mag_kf':float64[1/Å],
                  'kf':vector3[1/Å], 'dE':float64[meV], 'Q':vector3[1/Å],
                  'mag_Q':float64[1/Å], 'qx':float64[1/Å], 'qy':float64[1/Å],
                  'qz':float64[1/Å], 'en':float64[meV]})

# Calculate minimum and maximum kf

In [5]:
Lm = events.coords["Lm"]


def _calc_mag_kf_from_ef(ef):
    hbar = sc.constants.hbar
    m_n = sc.constants.neutron_mass
    kf = sc.sqrt(2 * m_n * ef) / hbar
    return kf.to(unit="1/angstrom")


monitor = ms.name_search("Monitor6", data)
tom_centroid = _calc_pulse_centroid(monitor)
vi = Lm / tom_centroid

ei = (0.5 * sc.constants.m_n * vi**2).to(unit="meV")

unit_ki = sc.vector([0, 0, 1])
mag_ki = ((sc.constants.neutron_mass * vi) / sc.constants.hbar).to(unit="1/angstrom")

ki = unit_ki * mag_ki


prop_ei = 0.8
max_ef = (1 + prop_ei) * ei
min_ef = (1 - prop_ei) * ei

min_kf = _calc_mag_kf_from_ef(min_ef)
max_kf = _calc_mag_kf_from_ef(max_ef)

print(f"min_kf: {min_kf}, max_kf: {max_kf}")

min_kf: <scipp.Variable> (tof: 1)    float64            [1/Å]  [0.595902], max_kf: <scipp.Variable> (tof: 1)    float64            [1/Å]  [1.78771]


# Access and calculate detector trajectory endpoints

In [6]:
sample_position = events.coords["sample_position"]
pixel_vec = scipp_object["positions"] - sample_position
pixel_vec = pixel_vec / sc.norm(pixel_vec)

Q_max = ki.to(unit="1/Å") - (pixel_vec * max_kf)
Q_min = ki.to(unit="1/Å") - (pixel_vec * min_kf)

kf_max = sc.broadcast(max_kf, dims=Q_max.dims, shape=Q_max.shape)
kf_min = sc.broadcast(min_kf, dims=Q_min.dims, shape=Q_min.shape)

kf_min

<scipp.Variable> (pixel_id: 30056, tof: 1)    float64           [1/Å]  [0.595902, 0.595902, ..., 0.595902, 0.595902]

# Calculate detector solid angles dOmega (detector counts/monitor counts)

In [7]:
scale_factor = (
    events.data.sum() / events.coords["monitor_counts"] * (4 * sc.constants.pi)
)
dOmega = (
    events.bins.sum().data / events.coords["monitor_counts"] * (4 * sc.constants.pi)
)
events.coords["dOmega"] = dOmega

# Generate 4D grid for histogramming

In [8]:
bins = generate_bins(
    qx=(-2, 1.5, 0.1),
    qy=(-0.1, 0.1),
    qz=(-1, 3.0, 0.1),
    en=(-0.2, 0.2),
)

# convert energy to kf

In [9]:
hbar = sc.constants.hbar
m_n = sc.constants.neutron_mass

kf_array = (
    (sc.sqrt(2 * m_n * (ei[0] - bins["en"])) / hbar)
    .to(unit="1/Å")
    .rename_dims({"en": "mag_kf"})
)
bins["mag_kf"] = kf_array[~sc.isnan(kf_array)]
bins["mag_kf"] = sc.sort(bins["mag_kf"], key="mag_kf")

edges = {key: bins[key] for key in ("qx", "qy", "qz", "mag_kf")}
display(edges)

{'qx': <scipp.Variable> (qx: 36)    float64           [1/Å]  [-2, -1.9, ..., 1.4, 1.5],
 'qy': <scipp.Variable> (qy: 2)    float64           [1/Å]  [-0.1, 0.1],
 'qz': <scipp.Variable> (qz: 41)    float64           [1/Å]  [-1, -0.9, ..., 2.9, 3],
 'mag_kf': <scipp.Variable> (mag_kf: 2)    float64           [1/Å]  [1.29575, 1.36822]}

# Calculate normalization factor per pixel

In [10]:
hbar = sc.constants.hbar
m_n = sc.constants.neutron_mass


def _calc_en_from_kf_endpoints(kf_in, kf_out):
    en = hbar**2 / m_n * 0.5 * ((kf_in**2 - kf_out**2) * sc.Unit("1/Å^2"))
    return sc.abs(en).to(unit="meV")

In [11]:
from voxel_traversal_4d import voxel_traversal_4d

# create norm_array
dims = list(edges.keys())
shape = [var.sizes[dim] - 1 for dim, var in edges.items()]

norm_arr = sc.zeros(
    dims=dims,
    shape=shape,
    unit=sc.Unit("meV^"),
)

edges_arr = [edge.values for edge in edges.values()]

# loop over all pixels

for pixel in range(Q_max.sizes["pixel_id"]):
    # calculate intersections
    start = np.concatenate(
        [Q_min["pixel_id", pixel].values[0], kf_min["pixel_id", pixel].values]
    )
    end = np.concatenate(
        [Q_max["pixel_id", pixel].values[0], kf_max["pixel_id", pixel].values]
    )
    for idx, p_in, p_out in voxel_traversal_4d(start, end, edges_arr):
        i, j, k, l = idx
        *_, kf_in = p_in
        *_, kf_out = p_out
        dE_i = _calc_en_from_kf_endpoints(kf_in, kf_out)

        # solid angle
        dOmega_i = dOmega["pixel_id", pixel]

        # sum
        norm_arr["qx", i]["qy", j]["qz", k]["mag_kf", l] += dE_i * dOmega_i

# Histogram data

In [12]:
%matplotlib widget

data_hist = sc.bin(events.data, **edges).hist()
display(data_hist)
pp.plot(
    data_hist.squeeze().transpose(),
    coords=["qx", "qz"],
    grid=True,
    cmap="turbo",
)

<scipp.DataArray>
Dimensions: Sizes[qx:35, qy:1, qz:40, mag_kf:1, ]
Coordinates:
* mag_kf                    float64           [1/Å]  (mag_kf [bin-edge])  [1.29575, 1.36822]
* qx                        float64           [1/Å]  (qx [bin-edge])  [-2, -1.9, ..., 1.4, 1.5]
* qy                        float64           [1/Å]  (qy [bin-edge])  [-0.1, 0.1]
* qz                        float64           [1/Å]  (qz [bin-edge])  [-1, -0.9, ..., 2.9, 3]
Data:
                            float64         [counts]  (qx, qy, qz, mag_kf)  [0, 0, ..., 0, 0]

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

# Data divided by normalization factor

In [13]:
data_hist.squeeze()

<scipp.DataArray>
Dimensions: Sizes[qx:35, qz:40, ]
Coordinates:
  mag_kf                    float64           [1/Å]  (mag_kf [bin-edge])  [1.29575, 1.36822]
* qx                        float64           [1/Å]  (qx [bin-edge])  [-2, -1.9, ..., 1.4, 1.5]
  qy                        float64           [1/Å]  (qy [bin-edge])  [-0.1, 0.1]
* qz                        float64           [1/Å]  (qz [bin-edge])  [-1, -0.9, ..., 2.9, 3]
Data:
                            float64         [counts]  (qx, qz)  [0, 0, ..., 0, 0]

In [14]:
norm_arr

<scipp.Variable> (qx: 35, qy: 1, qz: 40, mag_kf: 1)    float64            [meV]  [0, 0, ..., 0, 0]

In [15]:
# norm_factor = events.bins.concat().value.copy()
# norm_factor.data = sc.bins_like(events, events.coords["dOmega"]).bins.concat().value

# norm_hist = sc.bin(norm_factor, **edges).hist()

pp.plot(
    (data_hist.squeeze().transpose())/norm_arr.squeeze(),
    coords=["qx", "qz"],
    grid=True,
    cmap="turbo",
    vmin=0,
    vmax=5e5,
)

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…